# Confidence calibration

How well the model's stated `confidence` matches how often it is right, for one `evaluate` sweep.

After https://arxiv.org/pdf/2512.16030

In [24]:
%matplotlib ipympl

In [25]:
from analysis.build_data import load_trace_df
from analysis.calibration import (
    category_accuracy_table,
    category_table,
    confidence_metrics,
    expected_calibration_error,
    horizon_table,
    kalshibench_metrics,
)


In [26]:
# one scoreable forecast per row, from every child run of the sweep (see analysis/build_data.py)
forecasts = load_trace_df("outcome-confidence", run_name="2k-balanced")

In [27]:
forecasts

,cutoff,record_id,request_time,model,category,market_close,confidence,predicted_resolution,outcome_occurred,correct,predicted_yes,p_yes,confidence_bin
3995,2026-06-30,-8553816499929850732,2026-11-08 13:41:53.939,claude-fable-5-1,companies,2026-08-27 19:06:25,0.93,Yes,True,True,True,0.93,"(0.9, 1.0]"
3994,2026-06-30,2192373202815083630,2026-11-08 13:41:53.940,claude-fable-5-1,companies,2026-10-29 22:01:06,0.70,No,True,False,False,0.30,"(0.6, 0.7]"
3993,2026-06-30,-8747809543395057657,2026-11-08 13:41:53.940,claude-fable-5-1,companies,2026-10-29 14:24:42,0.93,No,True,False,False,0.07,"(0.9, 1.0]"
3990,2026-06-30,7242157369701813651,2026-11-08 13:41:53.941,claude-fable-5-1,companies,2026-08-27 19:06:25,0.85,Yes,True,True,True,0.85,"(0.8, 0.9]"
3992,2026-06-30,43863750322640146,2026-11-08 13:41:53.941,claude-fable-5-1,companies,2026-10-29 22:01:07,0.84,Yes,True,True,True,0.84,"(0.8, 0.9]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
547,2025-08-31,-4086636522688444279,2026-11-09 02:28:15.433,claude-opus-4-5,finance,2026-09-25 20:00:00,0.70,No,False,True,False,0.30,"(0.6, 0.7]"
523,2025-08-31,3027148837804025644,2026-11-09 02:28:18.994,claude-opus-4-5,finance,2026-09-17 20:00:00,0.52,Up,True,True,True,0.52,"(0.5, 0.6]"
546,2025-08-31,3607940872346059245,2026-11-09 02:28:21.912,claude-opus-4-5,finance,2026-09-24 20:00:00,0.52,No,True,False,False,0.48,"(0.5, 0.6]"
519,2025-08-31,-5764099732138419860,2026-11-09 02:28:22.177,claude-opus-4-5,finance,2026-09-09 21:00:00,0.72,No,True,False,False,0.28,"(0.7, 0.8]"


## Dataset statistics

Table 2 of the paper: the markets in the sample by category, with the share resolving to the priced outcome (see `category_table` in `analysis/calibration.py`).

In [8]:
category_table(forecasts).round(1)

,N,%,Yes%
category,,,
economics,413,20.6,40.4
companies,400,19.9,51.2
politics,400,19.9,25.0
finance,398,19.8,43.7
elections,397,19.8,28.2


## Expected calibration error

Forecasts binned by stated confidence (`confidence_bin`, from `load_trace_df`; see `expected_calibration_error` in `analysis/calibration.py`).

In [9]:
ece = expected_calibration_error(forecasts)

In [10]:
ece_by_model = forecasts.groupby("model").apply(
    expected_calibration_error, include_groups=False
)

## KalshiBench table

Table 4 of the paper, per model (see `kalshibench_metrics` in `analysis/calibration.py` for the columns).

In [11]:
model_table = forecasts.groupby("model").apply(
    kalshibench_metrics, include_groups=False
)

In [12]:
model_table.round(3)

,N,Acc,F1,F1yes,Brier,BSS,ECE,MCE
model,,,,,,,,
claude-fable-5-1,543.0,65.930,0.531,0.286,0.211,0.106,0.031,0.062
claude-opus-4-5,413.0,59.080,0.583,0.641,0.246,0.015,0.140,0.241
claude-opus-5-5,1606.0,70.237,0.626,0.457,0.182,0.194,0.021,0.064


## Confidence analysis

Table 5 of the paper, per model: average confidence, confidence when wrong, and the overconfidence rate above 70/80/90% (see `confidence_metrics` in `analysis/calibration.py`).

In [13]:
confidence_table = forecasts.groupby("model").apply(
    confidence_metrics, include_groups=False
)

In [14]:
confidence_table.round(1)

,Avg Conf,Conf_wrong,OCR@70,OCR@80,OCR@90
model,,,,,
claude-fable-5-1,64.1,59.1,12.7,11.5,8.8
claude-opus-4-5,73.1,68.9,33.2,20.0,16.7
claude-opus-5-5,70.6,62.2,13.1,8.5,5.9


In [15]:
# the reliability table behind the number: per bin, its size, conf(B_b) and acc(B_b)
reliability = forecasts.groupby("confidence_bin", observed=True).agg(
    n=("correct", "size"), conf=("confidence", "mean"), acc=("correct", "mean")
)

## Forecast horizon

Accuracy and confidence by days from the model's training cutoff to the market's close (see `horizon_table` in `analysis/calibration.py`).

In [16]:
horizon_table(forecasts).round(2)

N    Acc  Avg Conf   ECE
model                                                      
claude-fable-5-1 (30.0, 60.0]     18  88.89     79.33  0.16
                 (60.0, 90.0]     41  82.93     80.30  0.15
                 (90.0, 180.0]   482   63.9     62.13  0.04
                 (180.0, 365.0]    1    0.0     93.00  0.93
                 (365.0, inf]      1    0.0     70.00  0.70
claude-opus-4-5  (-0.001, 30.0]    5   60.0     73.80  0.31
                 (30.0, 60.0]     21  95.24     93.52  0.07
                 (60.0, 90.0]     34  61.76     81.97  0.20
                 (90.0, 180.0]    43   81.4     88.23  0.07
                 (180.0, 365.0]  283  53.71     68.30  0.15
                 (365.0, inf]     27  48.15     72.48  0.27
claude-opus-5-5  (-0.001, 30.0]  115  85.22     77.19  0.09
                 (30.0, 60.0]    212  76.89     75.58  0.05
                 (60.0, 90.0]    491   66.6     71.66  0.06
                 (90.0, 180.0]   777  68.73     67.55  0.02
                 (180.0, 365.0]    6   50.0     86.00  0.48
                 (365.0, inf]      5   60.0     70.20  0.41

## Market category

Accuracy and confidence by model and market category (see `category_accuracy_table` in `analysis/calibration.py`).

In [17]:
category_accuracy_table(forecasts).round(2)

N    Acc  Avg Conf   ECE
model            category                             
claude-fable-5-1 companies   19  68.42     73.32  0.18
                 economics  397  61.46     58.98  0.04
                 elections  127  79.53     78.91  0.06
claude-opus-4-5  companies  400  58.75     73.34  0.15
                 economics   13  69.23     66.31  0.03
claude-opus-5-5  companies   19  68.42     70.95  0.25
                 economics  394  61.42     57.21  0.05
                 elections  395  77.22     78.08  0.04
                 finance    398  63.32     67.88  0.05
                 politics   400   79.0     79.22  0.06